In [ ]:
import heapq
import time
from tabulate import tabulate

Mazes = {
  "MAP_A" : [
    "S..#....",
    "##.#.##.",
    "....~~..",
    ".####.#.",
    "...~..#G"
] ,

  "MAP_B" : [
    "S~~~~~~~G",
    ".#######.",
    ".........",
    ".#.#.#.#.",
    "........."
] ,

  "MAP_C" : [
    "S..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "....#..G"
] ,
  "MAP_D" : [
    "S..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "....#..."
] ,
  "MAP_E" : [
    "#..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "....#..G"
] ,
  "MAP_F" : [
    "S..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "..G.#..#"
]
   }

In [ ]:
def Neighbores(current_row,current_col,grid):
  valid_neighbores = []
  directions = [(0,1),(0,-1),(1,0),(-1,0)]
  for row_bound,col_bound in directions:
    new_row = current_row + row_bound
    new_col = current_col + col_bound

    # Checking the boundaries
    if (0 <= new_row < len(grid)) and (0 <= new_col < len(grid[0])):

        type_cell = grid[new_row][new_col]

        # Checking for walls
        if type_cell != '#':

            if type_cell == '~':
              cost = 3
            else:
              cost = 1

            valid_neighbores.append(((new_row,new_col),cost))
  return valid_neighbores



In [ ]:
def UCS(maze):

  start = None
  goal = None
  # Locating the Start and Goal positions
  for r in range(len(maze)):
    for c in range(len(maze[0])):
      if maze[r][c] == 'S':
        start = (r,c)
      elif maze[r][c] == 'G':
        goal = (r,c)
  if not start or not goal:
    return 'No path found'

  frontier = []
  heapq.heappush(frontier,(0,start))

  came_from = {start : None}
  total_current_cost = {start : 0}
  nodes_expanded = 0
  max_frontier_size = 1

  while len(frontier) > 0 :
    current_cost , current_node = heapq.heappop(frontier)
    nodes_expanded += 1

    # Stop when reach the Goal
    if current_node == goal:
      break

    # If not keep exploring the neighores
    for next_node,cost in Neighbores(current_node[0],current_node[1],maze):
      new_cost = total_current_cost[current_node] + cost

      if next_node not in total_current_cost or new_cost < total_current_cost[next_node]:
        total_current_cost[next_node] = new_cost
        heapq.heappush(frontier,(new_cost,next_node))
        total_current_cost[next_node] = new_cost
        came_from[next_node] = current_node

        # Updating the max frontier size
        max_frontier_size = max(max_frontier_size,len(frontier))

  if goal not in came_from :
    return 'No path found' , 0 ,nodes_expanded, max_frontier_size

  current = goal
  path= []

  while current != start :
    path.append(current)
    current = came_from[current]
  path.append(start)
  path.reverse()    # To get the start to goal path

  return path , total_current_cost[goal] , nodes_expanded , max_frontier_size

In [ ]:
def heuristic(node,goal):
  return abs(node[0] - goal[0]) + abs(node[1] - goal[1])


def A_Star(maze):

  # Locating the Start and Goal positions
  start = None
  goal = None
  for r in range(len(maze)):
    for c in range(len(maze[0])):
      if maze[r][c] == 'S':
        start = (r,c)
      elif maze[r][c] == 'G':
        goal = (r,c)
  if not start or not goal:
    return 'No path found'

  frontier = []
  heapq.heappush(frontier,(0,start))

  came_from = {start : None}
  total_current_cost = {start : 0}
  nodes_expanded = 0
  max_frontier_size = 1

  while len(frontier) > 0 :
    current_priority, current_node = heapq.heappop(frontier)
    nodes_expanded += 1

    # Stop when reach the Goal
    if current_node == goal:
      break

    for next_node,cost in Neighbores(current_node[0],current_node[1],maze):
      new_cost = total_current_cost[current_node] + cost

      if next_node not in total_current_cost or new_cost < total_current_cost[next_node]:
        total_current_cost[next_node] = new_cost

        # Total current cost + heuristic distance to the goal
        priority = new_cost + heuristic(next_node,goal)

        heapq.heappush(frontier,(priority,next_node))
        came_from[next_node] = current_node

        # Updating the max frontier size
        max_frontier_size = max(max_frontier_size,len(frontier))

  if goal not in came_from :
    return 'No path found' , 0 ,nodes_expanded, max_frontier_size

  current = goal
  path= []

  while current != start :
    path.append(current)
    current = came_from[current]
  path.append(start)
  path.reverse()

  return path , total_current_cost[goal] , nodes_expanded , max_frontier_size

In [ ]:
def display_path(path,maze):
  maze_list = [list(row) for row in maze]
  for row,col in path:
    maze_list[row][col] = '*'
  for row in maze_list:
    print(''.join(row))

In [ ]:
algorithms = [("UCS", UCS), ("A*", A_Star)]

table_data = []

for map_name, current_map in Mazes.items():
    print(f"\n{'='*40}\nExecuting on {map_name}\n{'='*40}")

    for algo_name, algo_func in algorithms:
        print(f"\n--- {algo_name} Algorithm ---")

        # Track execution time
        start_time = time.perf_counter()
        result = algo_func(current_map)
        end_time = time.perf_counter()

        run_time = end_time - start_time

        if isinstance(result, tuple):
            path_result, total_cost, expanded, max_front = result
        else:
            # Handles cases where start or goal is missing
            path_result = result
            total_cost = 0
            expanded = 0
            max_front = 0

        # Output based on whether a path exists
        if path_result == "No path found":
            print(f"Algorithm: {algo_name}")
            print("Path: No path found")
            print(f"Nodes Expanded: {expanded}")

            path_length = "N/A"
            cost_display = "N/A"
        else:
            path_length = len(path_result) - 1 # Subtract 1 because S is not counted
            cost_display = total_cost

            print(f"Algorithm: {algo_name}")
            print(f"Path: {path_result}")
            print(f"Total Path Cost: {total_cost}")
            print(f"Nodes Expanded: {expanded}")
            print("Map with Path:")
            display_path(path_result,current_map)

        # Store metrics
        table_data.append([
            algo_name,
            map_name,
            path_length,
            cost_display,
            expanded,
            max_front,
            f"{run_time:.6f}"
        ])

# Print the final comparison table
print("\n" + "="*80)
print("ALGORITHM PERFORMANCE COMPARISON")
print("="*80)
headers = ["Algorithm", "Map", "Path Length", "Path Cost", "Nodes Expanded", "Max Frontier", "Run Time (s)"]
print(tabulate(table_data, headers=headers, tablefmt="grid"))


Executing on MAP_A

--- UCS Algorithm ---
Algorithm: UCS
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Total Path Cost: 15
Nodes Expanded: 28
Map with Path:
***#....
##*#.##.
..******
.####.#*
...~..#*

--- A* Algorithm ---
Algorithm: A*
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Total Path Cost: 15
Nodes Expanded: 21
Map with Path:
***#....
##*#.##.
..******
.####.#*
...~..#*

Executing on MAP_B

--- UCS Algorithm ---
Algorithm: UCS
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (1, 8), (0, 8)]
Total Path Cost: 12
Nodes Expanded: 30
Map with Path:
*~~~~~~~*
*#######*
*********
.#.#.#.#.
.........

--- A* Algorithm ---
Algorithm: A*
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (1, 8), (0, 8)]
Total Path Cost: 12
Nodes Expanded: 15
Map with Path:
*~~~~~~~*
*#######*
********

* Both Uniform Cost Search (UCS) and A* Search successfully found the exact same optimal, cheapest path on the solvable maps.
* However, A* significantly outperformed UCS in efficiency by expanding far fewer nodes ,Because A* uses the Manhattan distance heuristic to guide the search toward the goal rather than exploring uniformly in all directions ,it evaluates less irrelevant terrain while still guaranteeing the cheapest route.
* For this warehouse delivery robot, A* is the clearly superior choice as it consistently computes the optimal path with fewer computational steps and faster execution times.